In [1]:
import pandas as pd
from tavily import TavilyClient
from openai import OpenAI
import os
from tqdm.auto import tqdm
from datetime import datetime

In [ ]:
TODAY = datetime.now().strftime("%Y-%m-%d")

TAVILY_API_KEY = ""     # 본인의 키 입력
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

tavily = TavilyClient(api_key=TAVILY_API_KEY)
client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

In [3]:
SYSTEM_PROMPT_STRICT = f"""
당신은 검색 결과에 기반한 팩트 검증 전문가입니다.
배경 지식을 사용하지 말고, 오직 제공된 [검색 결과]만을 근거로 답변하세요.

### [기준 정보]
- 오늘 날짜(TODAY): {TODAY}

### [엄격한 답변 규칙]
1. **근거 필수**: 검색 결과에 없는 내용은 절대 지어내지 마세요.
2. **정보 없음**: 검색 결과가 질문에 대한 명확한 답을 포함하지 않으면 "정보 없음"이라고만 출력하세요.
3. **단답형 선호**: 설명보다는 핵심 정답(수치, 단어, 날짜) 위주로 간결하게 답하세요.
4. **시간 계산**: '올해', '작년' 등의 표현은 TODAY를 기준으로 구체적인 연도로 바꿔서 이해하세요.
"""

In [4]:
def get_tavily_answer(question):
    try:
        search_result = tavily.search(
            query=question,
            search_depth="advanced", 
            max_results=5
        )
        
        context = "\n".join([
            f"[{i+1}] {r['content']} (출처: {r['url']})" 
            for i, r in enumerate(search_result['results'])
        ])
        
        response = client.chat.completions.create(
            model="solar-pro",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_STRICT},
                {"role": "user", "content": f"### [검색 결과]\n{context}\n\n### [질문]\n{question}"}
            ],
            temperature=0.0,
        )
        
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"Error on '{question}': {e}")
        return "Error"

In [6]:
df = pd.read_csv("/data/ephemeral/pro-nlp-finalproject-nlp-13/data/submission_combined.csv")
print(f"작업 전 '정보 없음' 개수: {len(df[df['model_response'] == '정보 없음'])}")

작업 전 '정보 없음' 개수: 846


In [9]:
import pandas as pd
import os
from tqdm.auto import tqdm
import time


BATCH_SIZE = 30  # 한 번에 처리하고 저장할 개수
INPUT_PATH = "/data/ephemeral/pro-nlp-finalproject-nlp-13/data/submission_combined.csv" 
OUTPUT_PATH = "/data/ephemeral/pro-nlp-finalproject-nlp-13/notebooks/Yoon/final_submission_strict.csv"
LOG_PATH = "/data/ephemeral/pro-nlp-finalproject-nlp-13/notebooks/Yoon/only_updated_rows.csv"


if os.path.exists(OUTPUT_PATH):
    print(f"🔄 기존 작업 파일 발견! 이어서 작업을 시작합니다: {OUTPUT_PATH}")
    df = pd.read_csv(OUTPUT_PATH)
else:
    print(f"🆕 새로운 작업을 시작합니다. 원본 파일 로드: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)


target_indices = df[df['model_response'] == "정보 없음"].index.tolist()
print(f"📌 총 처리해야 할 건수: {len(target_indices)}건")


total_updated = 0

for i in tqdm(range(0, len(target_indices), BATCH_SIZE), desc="Batch Processing"):
    batch_indices = target_indices[i : i + BATCH_SIZE]
    
    batch_updated_list = [] 
    
    for idx in batch_indices:
        row = df.loc[idx]
        
        try:
            new_answer = get_tavily_answer(row['question'])
            
            if "정보 없음" not in new_answer and "Error" not in new_answer:
                df.at[idx, 'model_response'] = new_answer
                batch_updated_list.append(df.loc[[idx]])
                
        except Exception as e:
            print(f"⚠️ Error at index {idx}: {e}")
            continue

    # 30개(한 배치)가 끝날 때마다 파일 저장
    

    df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    
    if batch_updated_list:
        batch_df = pd.concat(batch_updated_list)
        
        if not os.path.exists(LOG_PATH):
            batch_df.to_csv(LOG_PATH, index=False, encoding='utf-8-sig', mode='w')
        else:
            batch_df.to_csv(LOG_PATH, index=False, header=False, encoding='utf-8-sig', mode='a')
            
        total_updated += len(batch_updated_list)
    


print("="*30)
print("✅ 모든 작업이 완료되었습니다.")
print(f"📊 총 업데이트된 행 개수: {total_updated}")
print(f"📂 최종 파일: {OUTPUT_PATH}")
print(f"📂 변경 내역: {LOG_PATH}")

🆕 새로운 작업을 시작합니다. 원본 파일 로드: /data/ephemeral/pro-nlp-finalproject-nlp-13/data/submission_combined.csv
📌 총 처리해야 할 건수: 846건


Batch Processing:   0%|          | 0/29 [00:00<?, ?it/s]

Error on '대한민국의 코미디언 부부 중에 이혼한 커플이 있어?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '가장 최근에 진행된 대한민국 대법관 취임식은 언제인가요?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '현재 대한민국 국립발레단의 단장은 누구인가요?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '현재까지 한국에서 천만 관객을 돌파한 영화는 몇 편이야?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '삼성의 계열사는 총 몇 개야?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '서울에 미슐랭 2스타 음식점은 총 몇 개야?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '현재 CBS 음악FM 93.9㎒의 오전 9시 프로그램은 누가 진행해?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '현재 멕시코 대통령은 누구야?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '가장 저렴한 아이쿱자연드림 조합 회비가 얼마인지 알려줘.': 432 Client Error:  for url: https://api.tavily.com/search
Error on '김미려는 자녀가 몇 명이야?': 432 Client Error:  for url: https://api.tavily.com/search
Error on '한일이 공동으로 운용한 인공위성의 개발 기관이